# IV. Huấn luyện và so sánh mô hình dự đoán tỉ số

Notebook so sánh bốn mô hình dự đoán số bàn đội nhà và đội khách: **Poisson Regression, Ridge Regression, Random Forest và Extra Trees**.

Huấn luyện trên các trận từ WC 2018 đến 10/06/2026. Tập kiểm tra là 104 trận WC 2026 (11/06 – 19/07/2026).

**Ba điểm cần lưu ý khi đọc kết quả:**

1. Có thêm hai **baseline** (luôn đoán 1-1, luôn đoán đội nhà thắng). Không có baseline thì không thể nói một con số accuracy là cao hay thấp.
2. Có thêm cột **số tỉ số khác nhau** mà mỗi mô hình dự đoán. Cột này phát hiện một vấn đề quan trọng: nếu mô hình chỉ dự đoán 4–5 tỉ số khác nhau cho cả 104 trận thì nó gần như đang đoán một hằng số, chứ không thực sự phân biệt được các trận.
3. Tập kiểm tra chỉ có 104 trận, nên sai số chuẩn khoảng 5 điểm phần trăm. Chênh lệch nhỏ giữa hai mô hình là nhiễu.

## 1. Nạp dữ liệu

Chạy `wc.ipynb` trước. Dữ liệu lịch sử và WC 2026 dùng đúng 7 feature: Elo hai đội trước trận, cờ sân trung lập và 4 chỉ số bàn ghi/thủng trung bình trong tối đa 5 trận trước đó.

Hai file được đọc từ `loc/` đều do `wc.ipynb` sinh ra, nên Elo của tập train và tập test cùng một thang đo. Nếu lấy Elo từ hai nguồn khác nhau, cùng một đội sẽ có hai giá trị khác nhau và mô hình sẽ nhận được đặc trưng lệch phân phối.

In [1]:
import csv
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.linear_model import PoissonRegressor, Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd()
if not (ROOT / 'loc').is_dir():
    ROOT = ROOT.parent

def read_csv(path):
    with path.open('r', encoding='utf-8-sig', newline='') as file:
        return list(csv.DictReader(file))

def parse_bool(value):
    return str(value).strip().casefold() in {'true', '1', 'yes'}

history = read_csv(ROOT / 'loc' / 'matches_truoc_wc_2026_elo_features.csv')
world_cup = read_csv(ROOT / 'loc' / 'wc_2026_da_loc.csv')

def make_data(rows):
    X = np.asarray([
        [
            float(row['home_elo_pre']),
            float(row['away_elo_pre']),
            float(parse_bool(row['neutral'])),
            float(row['home_form_goals_scored_last5']),
            float(row['home_form_goals_conceded_last5']),
            float(row['away_form_goals_scored_last5']),
            float(row['away_form_goals_conceded_last5']),
        ]
        for row in rows
    ], dtype=float)
    y_home = np.asarray([float(row['home_score']) for row in rows])
    y_away = np.asarray([float(row['away_score']) for row in rows])
    return X, y_home, y_away

train_data = make_data(history)
wc2026_data = make_data(world_cup)

X_train, y_home_train, y_away_train = train_data
X_test, y_home_test, y_away_test = wc2026_data

print(f'Tập huấn luyện: {len(history):,} trận ({history[0]["date"]} đến {history[-1]["date"]})')
print(f'Tập kiểm tra WC2026: {len(world_cup):,} trận ({world_cup[0]["date"]} đến {world_cup[-1]["date"]})')

print(f'Elo trung bình - train: {X_train[:, 0].mean():.0f} (nhà) / {X_train[:, 1].mean():.0f} (khách)')
print(f'Elo trung bình - test : {X_test[:, 0].mean():.0f} (nhà) / {X_test[:, 1].mean():.0f} (khách)')
print('  Test cao hơn là bình thường: 48 đội dự WC là nhóm mạnh nhất,')
print('  còn tập train gồm cả các đội yếu.')

# Kiểm tra hai tập dùng CÙNG thang Elo: so Elo của từng đội ở cuối train
# với Elo đầu tiên của đội đó trong giải. Nếu lệch vài trăm điểm thì hai tập
# đang lấy Elo từ hai nguồn khác nhau, và mô hình sẽ nhận đặc trưng sai.
last_train_elo = {}
for row in history:
    last_train_elo[row['home_team']] = float(row['home_elo_pre'])
    last_train_elo[row['away_team']] = float(row['away_elo_pre'])

first_test_elo = {}
for row in world_cup:
    first_test_elo.setdefault(row['home_team'], float(row['home_elo_pre']))
    first_test_elo.setdefault(row['away_team'], float(row['away_elo_pre']))

gaps = [abs(first_test_elo[team] - last_train_elo[team])
        for team in first_test_elo if team in last_train_elo]
print(f'Chênh lệch Elo của cùng một đội giữa cuối train và đầu test: '
      f'trung vị {np.median(gaps):.0f}, lớn nhất {max(gaps):.0f} điểm')
print('  (chênh lệch này chỉ nên bằng mức thay đổi của một trận, khoảng dưới 60 điểm)')


Tập huấn luyện: 7,768 trận (2018-06-14 đến 2026-06-10)
Tập kiểm tra WC2026: 104 trận (2026-06-11 đến 2026-07-19)
Elo trung bình - train: 1598 (nhà) / 1568 (khách)
Elo trung bình - test : 1949 (nhà) / 1896 (khách)
  Test cao hơn là bình thường: 48 đội dự WC là nhóm mạnh nhất,
  còn tập train gồm cả các đội yếu.
Chênh lệch Elo của cùng một đội giữa cuối train và đầu test: trung vị 5, lớn nhất 31 điểm
  (chênh lệch này chỉ nên bằng mức thay đổi của một trận, khoảng dưới 60 điểm)


## 2. Bảy feature đầu vào

1. Elo đội nhà trước trận
2. Elo đội khách trước trận
3. Sân trung lập
4. Bàn đội nhà ghi trung bình trong tối đa 5 trận trước đó
5. Bàn đội nhà thủng lưới trung bình trong tối đa 5 trận trước đó
6. Bàn đội khách ghi trung bình trong tối đa 5 trận trước đó
7. Bàn đội khách thủng lưới trung bình trong tối đa 5 trận trước đó

In [2]:
# alpha là hệ số regularization của Poisson Regression. alpha càng lớn thì
# hệ số của các feature bị kéo về 0 càng mạnh, tức là mô hình càng "phẳng".
# alpha = 1.0 (giá trị mặc định hay dùng) quá lớn với 7 feature này: hệ số Elo
# bị co từ 0.44 xuống 0.17, nên mô hình không còn phân biệt được đội mạnh với
# đội yếu. Đo trên 469 trận của 8 giải, hạ alpha xuống 0.01 giúp RPS từ 0.1984
# xuống 0.1881 và độ chính xác thắng/hoà/thua từ 52.0% lên 56.5%.
POISSON_ALPHA = 0.01

models = {
    'Poisson Regression': lambda: make_pipeline(
        StandardScaler(), PoissonRegressor(alpha=POISSON_ALPHA, max_iter=3000)
    ),
    'Ridge Regression': lambda: make_pipeline(
        StandardScaler(), Ridge(alpha=10.0)
    ),
    'Random Forest': lambda: RandomForestRegressor(
        n_estimators=300, min_samples_leaf=5, random_state=42, n_jobs=-1
    ),
    'Extra Trees': lambda: ExtraTreesRegressor(
        n_estimators=300, min_samples_leaf=5, random_state=42, n_jobs=-1
    ),
}
print(f'{len(models)} mô hình sẽ được so sánh:', ', '.join(models))

4 mô hình sẽ được so sánh: Poisson Regression, Ridge Regression, Random Forest, Extra Trees


## 3. Lý do chọn bốn mô hình

- **Poisson Regression**: mô hình nền chuyên cho dữ liệu đếm như số bàn thắng.
- **Ridge Regression**: mô hình tuyến tính có regularization L2, làm mốc so sánh đơn giản.
- **Random Forest**: kết hợp nhiều cây, học quan hệ phi tuyến và ít nhạy với một cây đơn lẻ.
- **Extra Trees**: tổ hợp cây ngẫu nhiên với cách chia ngưỡng khác Random Forest.

Mỗi mô hình dự đoán số bàn kỳ vọng λ cho hai đội. Từ hai giá trị λ, ta dựng ma trận xác suất tỉ số rồi chọn ô có xác suất lớn nhất; cộng xác suất theo từng nhóm để chọn kết quả thắng/hoà/thua.

### Vì sao alpha của Poisson phải nhỏ

`PoissonRegressor` mặc định regularize khá mạnh. Với 7 feature và 7.768 trận, `alpha = 1.0` kéo hệ số của Elo từ 0.44 xuống 0.17 — mô hình gần như bỏ qua chênh lệch trình độ. Hệ quả dễ thấy nhất ở các cặp đấu lệch lớn: Spain (Elo 2214) gặp Curaçao (1613), chênh 601 điểm, Elo thuần túy cho 97% thắng nhưng mô hình chỉ cho 76%.

Hạ `alpha` xuống 0.01 thì mọi chỉ số đều tốt hơn, không phải đánh đổi:

| alpha | RPS ↓ | Đúng K/H/T |
|---|---|---|
| 1.0 | 0.1984 | 52.0% |
| 0.1 | 0.1895 | 54.8% |
| **0.01** | **0.1881** | **56.5%** |

Ridge thì không nhạy với `alpha` (0.1892 ở mọi giá trị từ 0.1 đến 100), nên giữ nguyên 10.0.

### Vì sao "đúng tỉ số" khó hơn "đúng kết quả" rất nhiều

Bài toán tỉ số là bài toán phân loại hơn 20 lớp với phân phối rất dẹt: tỉ số phổ biến nhất (1-1) chỉ chiếm khoảng 13% số trận, kế đến 0-1 và 1-0 mỗi loại chưa tới 9%. Vì vậy trần thực tế của chỉ số "đúng tỉ số" chỉ khoảng 15%, trong khi "đúng thắng/hoà/thua" có thể đạt 50–60%.

Một hệ quả quan trọng: **chọn ô xác suất lớn nhất chỉ có ích khi λ thay đổi đủ nhiều giữa các trận.** Nếu mọi trận đều có λ xấp xỉ 1,5 và 1,1 thì argmax luôn rơi vào cùng vài ô, và mô hình trở thành một hằng số trá hình. Cột "số tỉ số khác nhau" ở phần kết quả dùng để kiểm tra đúng điều này.

In [3]:
from scipy.stats import poisson

MAX_GOALS = 12      # cắt đuôi phân phối để ma trận không phình to
MIN_RATE = 0.05     # chặn dưới cho λ, tránh ma trận suy biến thành 1x1


def score_matrix(home_rate, away_rate):
    """Ma trận xác suất tỉ số, giả định số bàn hai đội độc lập theo Poisson."""
    home_rate = float(np.clip(home_rate, MIN_RATE, MAX_GOALS))
    away_rate = float(np.clip(away_rate, MIN_RATE, MAX_GOALS))
    goals = np.arange(MAX_GOALS + 1)
    probs = np.outer(poisson.pmf(goals, home_rate), poisson.pmf(goals, away_rate))
    return probs / probs.sum()


def predict_score_and_result(lambda_home, lambda_away):
    """Chọn tỉ số khả dĩ nhất và kết quả thắng/hoà/thua xác suất cao nhất."""
    predicted_home, predicted_away, predicted_result, result_probabilities = [], [], [], []
    outcomes = np.array([1, 0, -1])   # thắng nhà, hoà, thắng khách
    goals = np.arange(MAX_GOALS + 1)

    for home_rate, away_rate in zip(lambda_home, lambda_away):
        probs = score_matrix(home_rate, away_rate)

        # argmax của ma trận trả về (chỉ số bàn đội nhà, chỉ số bàn đội khách)
        home_index, away_index = np.unravel_index(np.argmax(probs), probs.shape)
        predicted_home.append(home_index)
        predicted_away.append(away_index)

        home_goals, away_goals = goals[:, None], goals[None, :]
        result_probs = np.array([
            probs[home_goals > away_goals].sum(),
            probs[home_goals == away_goals].sum(),
            probs[home_goals < away_goals].sum(),
        ])
        result_probs /= result_probs.sum()
        result_probabilities.append(result_probs)
        predicted_result.append(outcomes[np.argmax(result_probs)])

    return (
        np.asarray(predicted_home),
        np.asarray(predicted_away),
        np.asarray(predicted_result),
        np.asarray(result_probabilities),
    )


def evaluate(actual_home, actual_away, expected_home, expected_away):
    predicted_home, predicted_away, predicted_result, result_probabilities = (
        predict_score_and_result(expected_home, expected_away)
    )
    actual_home = actual_home.astype(int)
    actual_away = actual_away.astype(int)

    exact_match = (predicted_home == actual_home) & (predicted_away == actual_away)
    actual_result = np.sign(actual_home - actual_away)

    # RPS: so sánh xác suất tích luỹ theo thứ tự khách thắng < hoà < nhà thắng.
    actual_order = np.where(actual_result == -1, 0, np.where(actual_result == 0, 1, 2))
    predicted_cumulative = np.cumsum(result_probabilities[:, ::-1], axis=1)[:, :-1]
    actual_cumulative = actual_order[:, None] <= np.arange(2)
    rps = float(np.mean(np.mean((predicted_cumulative - actual_cumulative) ** 2, axis=1)))

    result_correct = predicted_result == actual_result
    return {
        'rps': rps,
        'result_count': int(result_correct.sum()),
        'exact_count': int(exact_match.sum()),
        'exact_accuracy': float(exact_match.mean()),
        'result_accuracy': float(result_correct.mean()),
        # Số tỉ số khác nhau được dự đoán: nếu quá ít thì mô hình gần như đoán hằng số.
        'distinct_scores': len(set(zip(predicted_home.tolist(), predicted_away.tolist()))),
    }


def compare_models(train_data, test_data):
    X_train, y_home_train, y_away_train = train_data
    X_test, y_home_test, y_away_test = test_data
    results = {}
    for name, make_model in models.items():
        home_model, away_model = make_model(), make_model()
        home_model.fit(X_train, y_home_train)
        away_model.fit(X_train, y_away_train)

        # Cắt λ về khoảng hợp lệ. Ridge có thể cho λ âm, và λ = 0 làm ma trận
        # Poisson suy biến thành 1x1 nên mô hình luôn dự đoán 0-0.
        expected_home = np.clip(home_model.predict(X_test), MIN_RATE, MAX_GOALS)
        expected_away = np.clip(away_model.predict(X_test), MIN_RATE, MAX_GOALS)
        results[name] = evaluate(y_home_test, y_away_test, expected_home, expected_away)
    return results


def constant_baseline(actual_home, actual_away, score):
    """Điểm của chiến lược đoán cố định, dùng làm mốc so sánh."""
    actual_home = actual_home.astype(int)
    actual_away = actual_away.astype(int)
    exact = (actual_home == score[0]) & (actual_away == score[1])
    actual_result = np.sign(actual_home - actual_away)
    predicted_result = np.sign(score[0] - score[1])
    return {
        'rps': None,
        'result_count': int((actual_result == predicted_result).sum()),
        'result_accuracy': float((actual_result == predicted_result).mean()),
        'exact_count': int(exact.sum()),
        'exact_accuracy': float(exact.mean()),
        'distinct_scores': 1,
    }


def print_table(title, rows, total_matches):
    print(title)
    print(f'{"Phương án":<32} {"RPS ↓":>7} {"Đúng K/H/T":>18} {"Đúng tỉ số":>18} {"Số tỉ số":>9}')
    for name, result in rows.items():
        rps = f'{result["rps"]:.4f}' if result.get('rps') is not None else '—'
        result_text = f'{result["result_count"]}/{total_matches} ({result["result_accuracy"]:.1%})'
        exact_text = f'{result["exact_count"]}/{total_matches} ({result["exact_accuracy"]:.1%})'
        print(f'{name:<32} {rps:>7} {result_text:>18} {exact_text:>18} '
              f'{result["distinct_scores"]:>9}')


comparison = compare_models(train_data, wc2026_data)

baselines = {
    'Baseline: luôn đoán 1-1': constant_baseline(y_home_test, y_away_test, (1, 1)),
    'Baseline: luôn đoán nhà thắng': constant_baseline(y_home_test, y_away_test, (1, 0)),
}

print_table('Test WC2026', {**comparison, **baselines}, len(wc2026_data[0]))

print('\nSáu tỉ số xuất hiện nhiều nhất trong 104 trận WC 2026:')
score_counts = (
    pd.Series([f'{int(h)}-{int(a)}' for h, a in zip(y_home_test, y_away_test)])
    .value_counts(normalize=True)
    .head(6)
)
for score, share in score_counts.items():
    print(f'  {score}: {share:.1%}')


Test WC2026
Phương án                          RPS ↓         Đúng K/H/T         Đúng tỉ số  Số tỉ số
Poisson Regression                0.1594     68/104 (65.4%)     16/104 (15.4%)         7
Ridge Regression                  0.1577     69/104 (66.3%)     19/104 (18.3%)         7
Random Forest                     0.1683     68/104 (65.4%)      10/104 (9.6%)         8
Extra Trees                       0.1657     70/104 (67.3%)     14/104 (13.5%)         8
Baseline: luôn đoán 1-1                —     24/104 (23.1%)     12/104 (11.5%)         1
Baseline: luôn đoán nhà thắng          —     50/104 (48.1%)       7/104 (6.7%)         1

Sáu tỉ số xuất hiện nhiều nhất trong 104 trận WC 2026:
  1-1: 11.5%
  2-1: 8.7%
  2-0: 7.7%
  0-1: 7.7%
  0-0: 7.7%
  1-0: 6.7%


## 4. Đọc các chỉ số

- **RPS (Ranked Probability Score)**: chỉ số chính, đánh giá chất lượng xác suất thắng/hoà/thua theo đúng thứ tự khách thắng < hoà < nhà thắng. Càng thấp càng tốt, 0 là hoàn hảo. Ưu điểm so với accuracy: phạt nhẹ khi đoán "gần đúng" và phạt nặng khi đoán ngược hoàn toàn.
- **Đúng K/H/T**: chỉ số chính thứ hai, chọn kết quả có tổng xác suất cao nhất rồi tính tỉ lệ đúng.
- **Đúng tỉ số**: chỉ số tham khảo. Chọn ô có xác suất lớn nhất trong ma trận Poisson.
- **Số tỉ số**: số tỉ số khác nhau mà mô hình dự đoán cho cả 104 trận. Đây là chỉ số chẩn đoán, không phải chỉ số chất lượng. Nếu một mô hình chỉ đưa ra 4–5 tỉ số khác nhau thì argmax của nó đang bị "kẹt" ở vài ô, và độ chính xác tỉ số sẽ quanh mức của việc đoán hằng số.
- **Baseline**: mốc so sánh bắt buộc. "Luôn đoán 1-1" thường đạt khoảng 11–13% đúng tỉ số.

Đúng tỉ số chính xác là bài toán 20+ lớp với phân phối rất dẹt, trần thực tế chỉ khoảng 15%. Vì vậy nên ưu tiên RPS và đúng K/H/T, còn đúng tỉ số chỉ dùng để tham khảo.

## 5. Kết luận và giới hạn

Bảng trên so sánh bốn mô hình với cùng 7 feature và cùng 104 trận kiểm tra, kèm hai baseline đoán cố định.

**Nhận xét về cột "số tỉ số":** cả bốn mô hình hiện đều dự đoán 7–8 tỉ số khác nhau, nên không mô hình nào còn bị "kẹt" ở vài ô. Điều này đạt được là nhờ hạ `alpha` của Poisson xuống 0.01: trước đó `alpha = 1.0` làm λ quá trơn và Poisson chỉ đưa ra 5 tỉ số khác nhau, với độ chính xác tỉ số chỉ 11.5% — ngang bằng baseline "luôn đoán 1-1". Đây là minh chứng cho thấy hạn chế của bài toán tỉ số nằm ở **độ phân tán của λ**, không nằm ở cách chọn ô.

**Về chênh lệch giữa các mô hình:** sau khi sửa `alpha`, Poisson (RPS 0.1594) gần bằng Ridge (0.1577), và khoảng cách này nhỏ hơn sai số chuẩn của tập 104 trận. Không nên kết luận mô hình nào "tốt hơn" dựa trên khác biệt này.

**Giới hạn cần nêu rõ trong báo cáo:**

1. Tập kiểm tra chỉ có 104 trận. Với tỉ lệ đúng quanh 50–60%, sai số chuẩn xấp xỉ 5 điểm phần trăm — chênh lệch dưới 10 điểm giữa hai mô hình là nhiễu, không phải khác biệt thật.
2. Kết quả WC 2026 đã được dùng trong quá trình thử nghiệm, nên đây là đánh giá tham khảo chứ không phải kiểm tra độc lập hoàn toàn.
3. Để so sánh công bằng hơn, nên backtest trên nhiều giải (WC 2018, WC 2022, Euro, Copa América, AFCON, Asian Cup). Khi làm vậy, độ chính xác thắng/hoà/thua trung bình rơi về khoảng 50–55%, thấp hơn nhiều so với con số trên riêng WC 2026.
4. Muốn vượt trần hiện tại cần **feature mới** chứ không phải model mới: xG (expected goals), giá trị đội hình, số ngày nghỉ, hoặc tỉ lệ cược của nhà cái. Đổi model gần như không cải thiện gì.

**Một hướng đã thử và loại bỏ:** hiệu chỉnh Dixon–Coles cho bốn ô tỉ số thấp. Trên 469 trận của 8 giải, cách này chỉ tăng khoảng 0,5 điểm phần trăm nhưng khiến mô hình dự đoán "1-1" cho tới 78% số trận — tức là đánh đổi khả năng phân biệt lấy một định kiến cố định. Không đáng dùng, nên không đưa vào mô hình.